⏱️ **Time required:** ~10 minutes | **Type:** Interactive tutorial

# Developer Experience

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/04_developer_experience.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/04_developer_experience.ipynb)

Structured diagnostics, DDL generation, DAG visualization, dry run previews, surgical resets, and multi-channel alerts — all from the contract.

In [1]:
# Install lakelogic
!pip install -q lakelogic[polars,duckdb]

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

lakelogic v1.21.0 | Local | c:\_Personal\_SaaS\lakelogic\examples\colab


### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb"  # 'polars' , 'spark'

---
## 1. Structured Diagnostics — Powered by `loguru`

**The Problem:** Your pipeline failed. The log says `ERROR: validation failed`. No contract name, no run ID, no timestamp with timezone. You grep through 50 log files.

**The Solution:** LakeLogic uses `loguru` for structured logging. Every line includes precise timestamps, severity levels, the exact function path, and execution tags — drastically cutting troubleshooting time.

In [2]:
# LakeLogic uses loguru out of the box — no logging config needed.
# Just run a processor and observe the structured log output.

contract = s.write_contract(
    """
version: 1.0.0
dataset: diagnostics_demo
model:
  fields:
    - name: id
      type: integer
      required: true
    - name: value
      type: string
quality:
  row_rules:
    - name: has_value
      sql: "value IS NOT NULL AND value != ''"
""",
    "04_developer_experience_demo/diag.yaml",
)

proc = ll.DataProcessor(contract, engine=ENGINE)
source_df = ll.DataGenerator(contract).generate(rows=50, invalid_ratio=0.1, output_format=ENGINE)
good, bad = proc.run(source_df)
good, bad = s.to_polars(good), s.to_polars(bad)

# ── Observe: every log line has timestamp | level | module:function:line ──
print(f"\nResult: good={len(good)}, bad={len(bad)}")
print("\n\u2705 Every log line above includes:")
print("   • ISO timestamp with timezone")
print("   • Severity level (INFO, WARNING, ERROR)")
print("   • Module path: lakelogic.core.processor:run:788")
print("   • Run metrics: Source count, Good/Quarantine split, ratio")

2026-04-28 06:49:37.568 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: diagnostics_demo
2026-04-28 06:49:37.568 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 45 valid + 5 invalid = 50 total
2026-04-28 06:49:37.568 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:49:37.569 | INFO     | lakelogic.core.generator:generate:3448 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-28 06:49:37.572 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 50 records built
2026-04-28 06:49:37.572 | INFO     | lakelogic.core.generator:generate:3504 -    Test cases : 6 across 4 categories
2026-04-28 06:49:37.573 | INFO     | lakelogic.core.generator:generate:3506 -      EMPTY_STRING                      2 injections
2026-04-28 06:49:37.573 | INFO     | lakelogic.core.generator:generate:3506 -      BOUNDARY_VALUE       


Result: good=46, bad=4

✅ Every log line above includes:
   • ISO timestamp with timezone
   • Severity level (INFO, WARNING, ERROR)
   • Module path: lakelogic.core.processor:run:788
   • Run metrics: Source count, Good/Quarantine split, ratio


---
## 2. DDL-Only Mode — Schema Migration Without Running a Pipeline

**The Problem:** You need to create the target table schema before the first pipeline run, but you don't want to process any data yet.

**The Solution:** `DataProcessor.generate_ddl()` generates CREATE TABLE DDL directly from the contract — perfect for CI/CD migrations.

In [3]:
import os
import shutil
import yaml
import polars as pl
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline

# ── Clean slate ──────────────────────────────────────────────────────
DDL_DIR = "./ddl_demo"
if os.path.exists(DDL_DIR):
    shutil.rmtree(DDL_DIR)
os.makedirs(f"{DDL_DIR}/contracts/bronze", exist_ok=True)
os.makedirs(f"{DDL_DIR}/landing/events", exist_ok=True)

# ── Contract ─────────────────────────────────────────────────────────
ddl_contract = s.write_contract(
    """
version: 1.0.0
dataset: user_events
info:
  title: bronze_user_events
  target_layer: bronze
source:
  type: landing
  path: "./ddl_demo/landing/events"
  format: ndjson
model:
  fields:
    - name: event_id
      type: integer
      required: true
    - name: user_id
      type: string
      required: true
    - name: event_type
      type: string
    - name: payload
      type: string
    - name: created_at
      type: string
      required: true
""",
    f"{DDL_DIR}/contracts/bronze/user_events_v1.0.yaml",
)

# ── Preview: show what DDL LakeLogic generates per backend ────────
proc = ll.DataProcessor(ddl_contract, engine=ENGINE)
print("DuckDB DDL:")
print(proc.generate_ddl(backend="duckdb"))
print("\nSpark DDL:")
print(proc.generate_ddl(backend="spark"))

# ── _system.yaml registry ───────────────────────────────────────────
system_yaml = {
    "domain": "demo",
    "system": "app",
    "contracts": [
        {
            "layer": "bronze",
            "entity": "user_events",
            "path": "contracts/bronze/user_events_v1.0.yaml",
            "enabled": True,
        },
    ],
    "environments": {
        "local": {
            "catalog": "local",
            "storage_root": "./ddl_demo/lakehouse",
            "data_root": "./ddl_demo/lakehouse",
            "quarantine_root": "./ddl_demo/lakehouse/_quarantine",
        }
    },
    "storage": {
        "external_location_root": "./ddl_demo/lakehouse",
    },
}

system_path = f"{DDL_DIR}/_system.yaml"
with open(system_path, "w") as f:
    yaml.dump(system_yaml, f, default_flow_style=False)

# ── Initialize Pipeline & run DDL-only ────────────────────────────
registry = DomainRegistry.from_yaml(system_path, environment="local", storage_mode="direct")
ddl_pipeline = LakehousePipeline(registry, engine=ENGINE)

summary = ddl_pipeline.run(
    target_layers="bronze",
    reset_layers="",
    reload_layers="",
    dry_run=False,
    entity_filter="",
    environment="local",
    parallel=False,
    max_workers=1,
    ddl_only=True,  # ← CREATE TABLES ONLY, NO DATA PROCESSING
    created_by="ddl_migration_demo",
)

print("\n" + "=" * 60)
print("DDL-ONLY RUN SUMMARY")
print("=" * 60)
print(f"  Pipeline run  : {summary.run_id}")
print("  DDL only      : True")
print()
for r in summary.results:
    table_name = r.get("table_name", r.get("contract", "?"))
    layer = r.get("layer", "?")
    status = r.get("status", "?")
    print(f"  [{layer:6s}] {table_name:30s} → {status}")
print("=" * 60)
print("\n✅ Tables created from contract — no data was processed.")

2026-04-28 06:49:37.691 | INFO     | lakelogic.pipeline.runner:run:1503 - Pipeline storage mode: direct
2026-04-28 06:49:37.693 | INFO     | lakelogic.core.ddl:_init_delta_table_from_contract:956 - No materialization.target_path configured — Delta table will be created on first data write.
2026-04-28 06:49:37.694 | INFO     | lakelogic.pipeline.runner:generate_ddl_only:992 - Table created for user_events


DuckDB DDL:
CREATE TABLE IF NOT EXISTS user_events (
  event_id INTEGER NOT NULL,
  user_id VARCHAR NOT NULL,
  event_type VARCHAR,
  payload VARCHAR,
  created_at VARCHAR NOT NULL
);

Spark DDL:
CREATE TABLE IF NOT EXISTS user_events (
  event_id INT NOT NULL,
  user_id STRING NOT NULL,
  event_type STRING,
  payload STRING,
  created_at STRING NOT NULL
)
USING DELTA
TBLPROPERTIES ('delta.enableDeletionVectors' = false);

DDL-ONLY RUN SUMMARY
  Pipeline run  : 9cdb4fdc-6668-4e0a-853b-af847c02c6f8
  DDL only      : True

  [bronze] user_events                    → ddl_created

✅ Tables created from contract — no data was processed.


---
## 3. DAG Dependency Viewer — Execution Order at a Glance

**The Problem:** You have 15 contracts with dependencies. Running them in the wrong order corrupts downstream tables.

**The Solution:** `LakehousePipeline.visualize_dag()` renders the full dependency graph from your `_system.yaml` registry — showing bronze → silver → gold flow and execution order.

In [4]:
import os
import yaml
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline
from IPython.display import HTML, display

# ── Create inline contracts ──────────────────────────────────────────
DAG_DIR = "./dag_demo"
os.makedirs(f"{DAG_DIR}/contracts/bronze", exist_ok=True)
os.makedirs(f"{DAG_DIR}/contracts/silver", exist_ok=True)

# Bronze: orders (raw landing)
s.write_contract(
    """
version: 1.0.0
dataset: orders
info:
  title: bronze_demo_orders
  target_layer: bronze
source:
  type: landing
  path: "./dag_demo/landing/orders"
  format: ndjson
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
""",
    f"{DAG_DIR}/contracts/bronze/bronze_demo_orders_v1.0.yaml",
)

# Bronze: customers (raw landing)
s.write_contract(
    """
version: 1.0.0
dataset: customers
info:
  title: bronze_demo_customers
  target_layer: bronze
source:
  type: landing
  path: "./dag_demo/landing/customers"
  format: ndjson
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
      pii: true
""",
    f"{DAG_DIR}/contracts/bronze/bronze_demo_customers_v1.0.yaml",
)

# Silver: customers_cleaned (depends on bronze customers + customers)
s.write_contract(
    """
version: 1.0.0
dataset: customers_cleaned
info:
  title: silver_demo_customers_cleaned
  target_layer: silver
source:
  type: table
  path: "./dag_demo/lakehouse/bronze/bronze_demo_customers"
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
      pii: true

""",
    f"{DAG_DIR}/contracts/silver/silver_demo_customers_v1.0.yaml",
)

# Silver: orders_cleaned (depends on bronze orders + customers)
s.write_contract(
    """
version: 1.0.0
dataset: orders_cleaned
info:
  title: silver_demo_orders_cleaned
  target_layer: silver
source:
  type: table
  path: "./dag_demo/lakehouse/bronze/bronze_demo_orders"
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float

downstream:
  - type: dashboard
    name: "Weekly Sales Performance"
    platform: power_bi
    url: "https://app.powerbi.com/..."
    owner: "marketing-analytics"

""",
    f"{DAG_DIR}/contracts/silver/silver_demo_orders_v1.0.yaml",
)

print("\u2705 Inline contracts created")

# ── Create inline _system.yaml registry ──────────────────────────────
system_yaml = {
    "domain": "demo",
    "system": "ecommerce",
    "external_sources": [
        {
            "name": "ecommerce demo API",
            "source_domain": "ecommerce Vendor",
            "catalog_path": "external_storage_path_or_api",
            "consumed_by": ["orders", "customers"],
        }
    ],
    "contracts": [
        {
            "layer": "bronze",
            "entity": "orders",
            "path": "contracts/bronze/bronze_demo_orders_v1.0.yaml",
            "enabled": True,
        },
        {
            "layer": "bronze",
            "entity": "customers",
            "path": "contracts/bronze/bronze_demo_customers_v1.0.yaml",
            "enabled": True,
        },
        {
            "layer": "silver",
            "entity": "customers_cleaned",
            "path": "contracts/silver/silver_demo_customers_v1.0.yaml",
            "enabled": True,
        },
        {
            "layer": "silver",
            "entity": "orders_cleaned",
            "path": "contracts/silver/silver_demo_orders_v1.0.yaml",
            "depends_on": ["customers_cleaned"],
            "enabled": True,
        },
    ],
    "environments": {
        "local": {
            "catalog": "local",
            "storage_root": "./dag_demo/lakehouse",
            "data_root": "./dag_demo/lakehouse",
            "quarantine_root": "./dag_demo/lakehouse/_quarantine",
        }
    },
    "storage": {
        "external_location_root": "./dag_demo/lakehouse",
        "log_path": "./dag_demo/lakehouse/_logs",
    },
    "materialization": {
        "bronze": {
            "strategy": "append",
            "format": "delta",
        },
        "silver": {
            "strategy": "merge",
            "format": "delta",
        },
    },
}

system_path = f"{DAG_DIR}/_system.yaml"
with open(system_path, "w") as f:
    yaml.dump(system_yaml, f, default_flow_style=False)
print(f"\u2705 _system.yaml written to {system_path}")

# ── Build pipeline and visualize ─────────────────────────────────────
registry = DomainRegistry.from_yaml(system_path, environment="local", storage_mode="direct")
pipeline = LakehousePipeline(registry, engine=ENGINE)

display(HTML(pipeline.visualize_dag()))

✅ Inline contracts created
✅ _system.yaml written to ./dag_demo/_system.yaml


---
## 4. Dry Run Mode — Preview Before You Commit

**The Problem:** You changed a transformation and want to see the execution plan before it touches production data.

**The Solution:** Run with `dry_run=True`. The pipeline walks every contract in topological order and logs what it **would** execute — but **skips all processing and writes**.

In [5]:
# ── Dry Run: uses the same inline pipeline from Section 3 ───────────
summary = pipeline.run(
    target_layers="bronze,silver",
    reset_layers="",
    reload_layers="",
    dry_run=True,  # <--- PREVIEW ONLY
    entity_filter="",
    environment="local",
    parallel=False,
    max_workers=4,
    ddl_only=False,
    created_by="developer_experience_demo",
    reprocess_from=None,
    reprocess_to=None,
    reprocess_column=None,
    reprocess_values=None,
    retry_attempts=3,
    retry_base_wait_seconds=2,
    entity_timeout_minutes=60,
    max_consecutive_failures=2,
)

print("\n" + "=" * 60)
print("DRY RUN SUMMARY")
print("=" * 60)
print(f"  Pipeline run  : {summary.run_id}")
print(f"  Environment   : {summary.environment}")
print(f"  Dry run       : {summary.dry_run}")
print()
for r in summary.results:
    contract = r.get("contract", "?")
    layer = r.get("layer", "?")
    status = r.get("status", "?")
    print(f"  [{layer:6s}] {contract:25s} -> {status}")
print("=" * 60)
print("\n✅ Every contract was evaluated but nothing was processed or written.")

2026-04-28 06:49:37.796 | INFO     | lakelogic.pipeline.runner:run:1503 - Pipeline storage mode: direct
2026-04-28 06:49:37.797 | INFO     | lakelogic.pipeline.runner:run:1604 - ── Processing Layer: BRONZE (2 contracts) ──
2026-04-28 06:49:37.837 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2067 -   ─────────────────────────────────────────────────────────
2026-04-28 06:49:37.839 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2068 -   📄 [bronze] orders | Contract: bronze_demo_orders v1.0.0
2026-04-28 06:49:37.839 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2071 - DRY RUN - skipping orders
2026-04-28 06:49:37.842 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2067 -   ─────────────────────────────────────────────────────────
2026-04-28 06:49:37.844 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2068 -   📄 [bronze] customers | Contract: bronze_demo_customers v1.0.0
2026-04-28 06:49:37.845 | INFO    


DRY RUN SUMMARY
  Pipeline run  : 88ea175c-8581-44af-a2d2-891a4b475db4
  Environment   : local
  Dry run       : True

  [bronze] orders                    -> dry_run
  [bronze] customers                 -> dry_run
  [silver] customers_cleaned         -> skipped_no_upstream
  [silver] orders_cleaned            -> skipped_no_upstream

✅ Every contract was evaluated but nothing was processed or written.


---
## 5. Surgical Reset & Reload — Silver Only, Bronze Untouched

**The Problem:** Your Silver transformation logic had a bug. You need to re-run Silver without reprocessing Bronze (which took 4 hours).

**The Solution:** Reset and reload just the Silver layer. Bronze output stays untouched — simply re-feed Bronze data into the fixed Silver processor.

In [6]:
import os
import shutil
import yaml
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline

# ── Clean Slate ──────────────────────────────────────────────────────
SUR_DIR = "./sur_demo"
if os.path.exists(SUR_DIR):
    shutil.rmtree(SUR_DIR, ignore_errors=True)
os.makedirs(f"{SUR_DIR}/contracts/bronze", exist_ok=True)
os.makedirs(f"{SUR_DIR}/contracts/silver", exist_ok=True)

# ── Bronze Contract: orders ──────────────────────────────────────────
s.write_contract(
    """
version: 1.0.0
dataset: orders
info:
  title: bronze_sur_orders
  table_name: bronze_sur_orders
  target_layer: bronze
source:
  type: landing
  path: "./sur_demo/landing/orders"
  format: ndjson
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
""",
    f"{SUR_DIR}/contracts/bronze/bronze_sur_orders_v1.0.yaml",
)

# ── Silver Contract: orders_cleaned ──────────────────────────────────
s.write_contract(
    """
version: 1.0.0
dataset: orders_cleaned
info:
  title: silver_sur_orders_cleaned
  table_name: silver_sur_orders_cleaned
  target_layer: silver
source:
  type: table
  path: "./sur_demo/lakehouse/bronze_sur_orders"
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
""",
    f"{SUR_DIR}/contracts/silver/silver_sur_orders_cleaned_v1.0.yaml",
)

print("✅ Contracts created")

# ── _system.yaml ─────────────────────────────────────────────────────
system_yaml = {
    "domain": "demo",
    "system": "surgical",
    "storage": {
        "external_location_root": "./sur_demo/lakehouse",
        "log_path": "./sur_demo/lakehouse/_logs",
    },
    "materialization": {
        "bronze": {
            "strategy": "append",
            "format": "delta",
        },
        "silver": {
            "strategy": "merge",
            "format": "delta",
        },
    },
    "contracts": [
        {
            "layer": "bronze",
            "entity": "orders",
            "path": "contracts/bronze/bronze_sur_orders_v1.0.yaml",
            "enabled": True,
        },
        {
            "layer": "silver",
            "entity": "orders_cleaned",
            "path": "contracts/silver/silver_sur_orders_cleaned_v1.0.yaml",
            "depends_on": ["orders"],
            "enabled": True,
        },
    ],
    "environments": {
        "local": {
            "catalog": "local",
            "storage_root": "./sur_demo/lakehouse",
            "data_root": "./sur_demo/lakehouse",
            "quarantine_root": "./sur_demo/lakehouse/_quarantine",
        }
    },
}

system_path = f"{SUR_DIR}/_system.yaml"
with open(system_path, "w") as f:
    yaml.dump(system_yaml, f, default_flow_style=False)
print(f"✅ _system.yaml written to {system_path}")

# ── Build Pipeline ───────────────────────────────────────────────────
registry = DomainRegistry.from_yaml(system_path, environment="local", storage_mode="direct")
sur_pipeline = LakehousePipeline(registry, engine=ENGINE)

# ── Generate Landing Data ────────────────────────────────────────────
os.makedirs("./sur_demo/landing/orders", exist_ok=True)
pl.DataFrame({"order_id": [1, 2, 3], "amount": [10.5, 20.0, 35.0]}).write_ndjson(
    "./sur_demo/landing/orders/data.ndjson"
)
print("✅ Landing data generated")

# ── RUN 1: Full Pipeline (Bronze + Silver) ───────────────────────────
print("\n── RUN 1: Full Pipeline (Bronze → Silver) ──")
summary_1 = sur_pipeline.run(
    target_layers="bronze,silver",
    reset_layers="",
    reload_layers="",
    dry_run=False,
    entity_filter="",
    environment="local",
    parallel=False,
    max_workers=1,
    ddl_only=False,
    created_by="surgical_reset_demo",
    retry_attempts=1,
    retry_base_wait_seconds=2,
    entity_timeout_minutes=60,
    max_consecutive_failures=2,
)
print(f"✅ Run 1 done: {[r['contract'] for r in summary_1.results if r['status'] == 'success']}")

# ── RUN 2: Surgical Reset & Reload (Silver Only) ────────────────────
# Scenario: you fixed a transformation bug. You need Silver re-processed
#           from the existing Bronze output — WITHOUT rerunning Bronze.
print("\n── RUN 2: Surgical Reset & Reload (Silver Only) ──")
summary_2 = sur_pipeline.run(
    target_layers="silver",  # ← TARGET ONLY SILVER
    reset_layers="silver",  # ← WIPE SILVER TARGETS
    reload_layers="silver",  # ← IGNORE WATERMARKS
    dry_run=False,
    entity_filter="",
    environment="local",
    parallel=False,
    max_workers=1,
    ddl_only=False,
    created_by="surgical_reset_demo",
    retry_attempts=1,
    retry_base_wait_seconds=2,
    entity_timeout_minutes=60,
    max_consecutive_failures=2,
)
print(f"✅ Run 2 done: {[r['contract'] for r in summary_2.results if r['status'] == 'success']}")

print("\n✅ Silver was wiped and re-processed from Bronze output.")
print("✅ Bronze layer was completely untouched.")

2026-04-28 06:49:37.895 | INFO     | lakelogic.pipeline.runner:run:1503 - Pipeline storage mode: direct
2026-04-28 06:49:37.897 | INFO     | lakelogic.pipeline.runner:run:1604 - ── Processing Layer: BRONZE (1 contracts) ──
2026-04-28 06:49:37.897 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2067 -   ─────────────────────────────────────────────────────────
2026-04-28 06:49:37.900 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2068 -   📄 [bronze] orders | Contract: bronze_sur_orders v1.0.0
2026-04-28 06:49:37.902 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\colab\sur_demo\landing\orders via polars
2026-04-28 06:49:37.908 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=bronze] | Source: 3 | Total: 3 | Good: 3 | Quarantine: 0 | Ratio: 0.00%
2026-04-28 06:49:37.945 | INFO     | lakelogic.core.materialization:materialize_dataframe:3368 - 
Casting 2 column(s) to match cont

✅ Contracts created
✅ _system.yaml written to ./sur_demo/_system.yaml
✅ Landing data generated

── RUN 1: Full Pipeline (Bronze → Silver) ──


2026-04-28 06:49:37.961 | INFO     | lakelogic.core.materialization:materialize_dataframe:3557 - Materialized 3 rows to Delta table: sur_demo\lakehouse\bronze_sur_orders (strategy=append, mode=append)
2026-04-28 06:49:37.962 | INFO     | lakelogic.pipeline.runner:run:1604 - ── Processing Layer: SILVER (1 contracts) ──
2026-04-28 06:49:37.964 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2067 -   ─────────────────────────────────────────────────────────
2026-04-28 06:49:37.967 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2068 -   📄 [silver] orders_cleaned | Contract: silver_sur_orders_cleaned v1.0.0
2026-04-28 06:49:37.970 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\colab\sur_demo\lakehouse\bronze_sur_orders via polars
2026-04-28 06:49:37.978 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=silver] | Source: 3 | Total: 3 | Good: 3 | Quarantine: 0 | Ratio: 0.00%
2026-

✅ Run 1 done: ['orders', 'orders_cleaned']

── RUN 2: Surgical Reset & Reload (Silver Only) ──
✅ Run 2 done: ['orders_cleaned']

✅ Silver was wiped and re-processed from Bronze output.
✅ Bronze layer was completely untouched.


---
## 6. Multi-Channel Alerts — Slack, Teams, Email, Webhooks

**The Problem:** A quality breach fires at 2am. Nobody sees the email until 9am. Seven hours of bad data in production.

**The Solution:** LakeLogic’s `proc.notify()` dispatches alerts to Slack, Teams, email, or any webhook — powered by Apprise with Jinja2 template support. The notification config lives in the contract.

### Real Slack Alerts from LakeLogic

| SLO Breach Alert | Quarantine Alert |
|:---:|:---:|
| ![SLO Breach](assets/slack_slo_alert.png) | ![Quarantine Alert](assets/slack_quarantine_alert.png) |

> **These are real notifications** fired by LakeLogic during a pipeline run. The contract defines which channels receive which event types.

### Local Testing (Dry Run) 

LakeLogic includes a `type: "console"` notification adapter that renders payloads directly to stdout. This is perfect for local development and notebook testing, allowing you to preview how templates render without configuring external credentials.

In [7]:
# == Notifications in Action (Console / Local Testing) ==
# LakeLogic includes a `console` notification adapter that simply prints
# the rendered notification payload to standard output.
# This is perfect for verifying notification payloads locally.

import os
import shutil
import yaml
import polars as pl
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline

NOTIFY_DIR = "./notify_demo"
if os.path.exists(NOTIFY_DIR):
    shutil.rmtree(NOTIFY_DIR, ignore_errors=True)

os.makedirs(f"{NOTIFY_DIR}/contracts/bronze", exist_ok=True)

# 1. Bronze Contract with a quality rule that will quarantine some rows
s.write_contract(
    """
version: 1.0.0
dataset: notification_test
info:
  title: bronze_notify_test
  table_name: bronze_notify_test
  target_layer: bronze
source:
  type: landing
  path: "./notify_demo/landing/data"
  format: ndjson
model:
  fields:
    - name: id
      type: integer
      required: true
quality:
  row_rules:
    - name: positive_id
      sql: "id > 1"
quarantine:
  enabled: true
""",
    f"{NOTIFY_DIR}/contracts/bronze/bronze_notify_test_v1.0.yaml",
)

# 2. System registry with console notifications configured
system_yaml = {
    "domain": "demo",
    "system": "notifications",
    "storage": {
        "external_location_root": "./notify_demo/lakehouse",
    },
    "materialization": {
        "bronze": {"strategy": "append", "format": "delta"},
    },
    "notifications": [{"type": "console", "on_events": ["quarantine", "failure"]}],
    "contracts": [
        {
            "layer": "bronze",
            "entity": "notification_test",
            "path": "contracts/bronze/bronze_notify_test_v1.0.yaml",
            "enabled": True,
        },
    ],
    "environments": {
        "local": {
            "catalog": "local",
            "storage_root": "./notify_demo/lakehouse",
        }
    },
}

system_path = f"{NOTIFY_DIR}/_system.yaml"
with open(system_path, "w") as f:
    yaml.dump(system_yaml, f, default_flow_style=False)

# 3. Generate landing data (id=1 will be quarantined by the positive_id rule)
os.makedirs(f"{NOTIFY_DIR}/landing/data", exist_ok=True)
pl.DataFrame({"id": [1, 2, 3]}).write_ndjson(f"{NOTIFY_DIR}/landing/data/data.ndjson")

# 4. Run Pipeline -- quarantine fires the console notification
registry = DomainRegistry.from_yaml(system_path, environment="local", storage_mode="direct")
pipeline = LakehousePipeline(registry, engine=ENGINE)

print("\n-- Running Pipeline with Console Notifications --")
summary = pipeline.run(
    target_layers="bronze",
    environment="local",
    dry_run=False,
)
print("\n[OK] Note the [LAKELOGIC NOTIFICATION] blocks printed to the console above!")

2026-04-28 06:49:38.050 | INFO     | lakelogic.pipeline.runner:run:1503 - Pipeline storage mode: direct



-- Running Pipeline with Console Notifications --


2026-04-28 06:49:38.050 | INFO     | lakelogic.pipeline.runner:run:1604 - ── Processing Layer: BRONZE (1 contracts) ──
2026-04-28 06:49:38.050 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2067 -   ─────────────────────────────────────────────────────────
2026-04-28 06:49:38.051 | INFO     | lakelogic.pipeline.runner:_process_single_contract:2068 -   📄 [bronze] notification_test | Contract: bronze_notify_test v1.0.0
2026-04-28 06:49:38.054 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\colab\notify_demo\landing\data via polars
2026-04-28 06:49:38.058 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=bronze] | Source: 3 | Total: 3 | Good: 2 | Quarantine: 1 | Ratio: 33.33%
2026-04-28 06:49:38.094 | INFO     | lakelogic.core.materialization:materialize_dataframe:3368 - 
Casting 1 column(s) to match contract types before Delta write (new table):
Column Name                         | From Type


[LAKELOGIC NOTIFICATION]
Subject: [LOCAL] Quarantine Alert
------------------------------------------------------------
**LakeLogic** — bronze_notify_test (v1.0.0)


*⚠️ Quarantine Alert*

LakeLogic Alert: 1 records quarantined in 'bronze_notify_test'. Total (post-transform): 3 (ratio 33.33%)

Rule Failure Breakdown (records can fail multiple rules):
- Rule failed: positive_id (id > 1) (1)

• *Run ID:* `dba62f35-f35a-45ad-ae4e-48ee496e111d`
• *Pipeline Run:* `9143339d-f397-400a-a728-88c54668e005`
• *Engine:* polars
• *Source:* `C:\_Personal\_SaaS\lakelogic\examples\colab\notify_demo\landing\data`
• *Time (UTC):* 2026-04-28T05:49:38.059187+00:00
• *Table:* `bronze_notify_test`
• *Layer:* bronze




---
Run ID: `dba62f35-f35a-45ad-ae4e-48ee496e111d` | Engine: polars | 2026-04-28T05:49:38.059187+00:00



[OK] Note the [LAKELOGIC NOTIFICATION] blocks printed to the console above!


In [8]:
# ── Contract with notification config ────────────────────────────────
# In production, these URLs would be real webhook endpoints.
# For this demo, we show what the notification system produces.

alert_yaml = """
version: 1.0.0
dataset: alert_demo

ownership:
  contacts:
    - name: Oncall Engineer
      role: owner
      email: oncall@company.com

model:
  fields:
    - name: metric_id
      type: integer
      required: true
    - name: value
      type: float

quality:
  row_rules:
    - name: positive_value
      sql: "value > 0"

quarantine:
  enabled: true
  notifications:
    - type: slack
      target: https://hooks.slack.com/services/T00/B00/demo
      on_events: [failure, slo_breach, quarantine]
    - type: teams
      target: https://outlook.webhook.office.com/demo
      on_events: [failure]
    - type: webhook
      target: https://api.pagerduty.com/v2/enqueue
      on_events: [slo_breach]
"""

print("\u2500" * 60)
print("NOTIFICATION CONFIG (from contract)")
print("\u2500" * 60)
print(alert_yaml.strip())

# ── Simulate what the notification payload looks like ───────────────
import json
from datetime import datetime, timezone

payload = {
    "event": "dataset_quality_check",
    "subject": "[LOCAL] demo/ecommerce: Dataset Quality Check Alert",
    "message": "alert_demo: quarantine rate 12% exceeds SLO threshold of 5%",
    "run_id": "abc-1234-def-5678",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "contract": "alert_demo v1.0.0",
    "engine": "polars",
    "channels": ["slack", "teams", "pagerduty"],
}

print("\n" + "\u2500" * 60)
print("NOTIFICATION PAYLOAD (what Slack/Teams/Webhooks receive)")
print("\u2500" * 60)
print(json.dumps(payload, indent=2))

print("\n\u2705 One contract config → Slack + Teams + PagerDuty simultaneously.")
print("   In production, proc.notify('failure', 'message') dispatches to all channels.")

────────────────────────────────────────────────────────────
NOTIFICATION CONFIG (from contract)
────────────────────────────────────────────────────────────
version: 1.0.0
dataset: alert_demo

ownership:
  contacts:
    - name: Oncall Engineer
      role: owner
      email: oncall@company.com

model:
  fields:
    - name: metric_id
      type: integer
      required: true
    - name: value
      type: float

quality:
  row_rules:
    - name: positive_value
      sql: "value > 0"

quarantine:
  enabled: true
  notifications:
    - type: slack
      target: https://hooks.slack.com/services/T00/B00/demo
      on_events: [failure, slo_breach, quarantine]
    - type: teams
      target: https://outlook.webhook.office.com/demo
      on_events: [failure]
    - type: webhook
      target: https://api.pagerduty.com/v2/enqueue
      on_events: [slo_breach]

────────────────────────────────────────────────────────────
NOTIFICATION PAYLOAD (what Slack/Teams/Webhooks receive)
─────────────────────

## What You Just Saw

- **Structured diagnostics** — `loguru` powers every log line with timestamps, severity, and function paths
- **DDL generation** — `proc.generate_ddl()` creates CREATE TABLE DDL for any backend from the contract
- **DAG viewer** — `pipeline.visualize_dag()` renders the full dependency graph from `_system.yaml`
- **Dry run mode** — `pipeline.run(dry_run=True)` previews execution without processing or writing
- **Surgical reset** — reload Silver without re-ingesting Bronze
- **Multi-channel alerts** — Slack, Teams, email, webhooks from one contract config block

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, backfill |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Diagnostics, DDL, DAG viewer, dry run, surgical resets, alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.